# Mini-SGLang: Terminal PDB Real Trace Tutorial

This notebook does **not** run a model, initialize CUDA, or occupy a GPU. It is the guided companion for the real PDB trace running in your SSH terminal.

```text
Browser notebook: read the checkpoint and answer the question
SSH terminal PDB: run the real mini-sglang process and type where / p / n / s / r / c
```

`trace_first.py` installs temporary wrappers in the same Python process, pauses at a real function boundary, and then calls the original mini-sglang method. Thus `s` enters the real source under:

```text
/root/workspace/mini-sglang/python/minisgl/...
```

It is not a mock and not a hand-written scheduler.

Do not start an LLM from this notebook. The terminal trace owns the GPU.


## 0. Start the real terminal trace

If your SSH terminal already shows `(Pdb)`, do not start another copy. Otherwise run:

```bash
cd /root/workspace/mini-sglang

PATH=/root/miniconda3/bin:$PATH \
PYTHONPATH=/root/workspace/mini-sglang/python \
OMP_NUM_THREADS=1 \
MKL_NUM_THREADS=1 \
python trace_first.py --max-tokens 2
```

Keep that terminal open. Model loading finishes before the first trace stop.

### PDB commands

| Command | Meaning |
|---|---|
| `where` or `w` | Print the call stack |
| `l` | Show nearby source |
| `p expr` | Print an expression |
| `pp expr` | Pretty-print an object |
| `n` | Next line, do not enter a call |
| `s` | Step into the next Python call |
| `r` | Run until the current function returns |
| `c` | Continue to the next prepared trace stop |
| `q` | Quit this trace |

New-user rule: do not blindly `s` through `model.forward()`. PDB can show Python scheduler and Engine code, but cannot step into FlashInfer or CUDA device kernels.


In [1]:
# This tutorial has no torch, CUDA, LLM, or mini-sglang imports.
progress = {"passed": 0}


def require(previous):
    if progress["passed"] != previous:
        raise RuntimeError(f"Locked: finish checkpoint {previous + 1} first.")


def submit(checkpoint, question, keywords):
    require(checkpoint - 1)
    answer = input(question + "\n> ").strip().lower()
    if not any(word in answer for word in keywords):
        raise RuntimeError(
            "Not enough yet. Return to the PDB terminal, run the listed command, then retry."
        )
    progress["passed"] = checkpoint
    print(f"Checkpoint {checkpoint} passed. In the terminal, type c for the next stop.")

print("Tutorial ready. This notebook has not initialized a GPU.")


Tutorial ready. This notebook has not initialized a GPU.


## Checkpoint 1: `LLM.generate`

Terminal output should include:

```text
TRACE: LLM.generate: request enters offline scheduler
(Pdb)
```

In the terminal run:

```text
where
p args
p args[1]
p args[2]
```

Observe that `args[1]` is the prompt list and `args[2]` is `SamplingParams`.

To confirm the wrapper delegates to the real source:

```text
n
s
where
```

You will enter real `minisgl/llm/llm.py`. Read a few lines, run `r`, then stop here and answer below.

**Question: Is this an offline request from `LLM.generate`, or an online API-server request?**


In [ ]:
submit(1, "Checkpoint 1: answer offline or online API.", ["offline"])


## Checkpoint 2: `Scheduler._process_one_msg`

Terminal output should include:

```text
TRACE: Scheduler._process_one_msg: UserMsg arrives
message = UserMsg, uid = 0
input_ids.shape = (...,)
(Pdb)
```

Run:

```text
where
p msg
p type(msg)
p msg.input_ids.shape
s
l
n
n
p self.prefill_manager.pending_list
```

You are now in real `scheduler.py`: the `UserMsg` branch adds the request to `PrefillManager`.

Use `r`, then stop here and answer below.

**Question: after this returns, does the request first enter `pending_list` or `decode_manager.running_reqs`?**


In [ ]:
submit(2, "Checkpoint 2: answer pending or decode running.", ["pending"])


## Checkpoint 3: `PrefillManager.schedule_next_batch`

Terminal output should include:

```text
TRACE: PrefillManager.schedule_next_batch
prefill_budget = 256
pending = [0]
(Pdb)
```

Run:

```text
p self.pending_list
p self.pending_list[0].input_len
p self.pending_list[0].output_len
s
```

In real `prefill.py`, observe the path:

```text
PrefillAdder -> try_add_one -> Req -> Batch(phase="prefill")
```

Read the short method, then use `r`.

**Question: why is the first batch prefill rather than decode: does it process original prompt/input tokens, or already-generated continuation tokens?**


In [ ]:
submit(3, "Checkpoint 3: answer prompt or input tokens.", ["prompt", "input"])


## Checkpoint 4: `Scheduler._prepare_batch`

Terminal output should include:

```text
TRACE: Scheduler._prepare_batch: make GPU metadata
phase = prefill
requests(uid, cached_len, device_len, remain_len) = ...
(Pdb)
```

Run:

```text
p batch.phase
p batch.reqs
p [(r.uid, r.cached_len, r.device_len, r.remain_len) for r in batch.reqs]
s
l
n
n
n
n
n
p batch.positions.shape
p batch.out_loc.shape
```

The real code assembles:

```text
pad_batch -> allocate_paged -> positions -> input/write mapping -> out_loc -> attention metadata
```

Do not print full KV cache tensors. Use `r` when ready.

**Question: is this step mainly generating text, or building GPU/attention locations and metadata?**


In [ ]:
submit(4, "Checkpoint 4: answer metadata, address, or location.", ["metadata", "address", "location", "位置"])


## Checkpoint 5: `Scheduler._forward` and `Engine.forward_batch`

First terminal stop:

```text
TRACE: Scheduler._forward: invoke Engine
phase = prefill
(Pdb)
```

Run:

```text
p forward_input.batch.phase
p forward_input.batch.input_ids.shape
s
```

You enter real `scheduler.py::_forward`. Observe:

```text
batch.input_ids = self.token_pool[input_mapping]
forward_output = self.engine.forward_batch(...)
self.token_pool[output_mapping] = forward_output.next_tokens_gpu
```

At `self.engine.forward_batch(...)`, type `s`. You then hit:

```text
TRACE: Engine.forward_batch: eager model forward + sampling
(Pdb)
```

Enter real `engine.py`, then run:

```text
l
n
n
p batch.phase
p batch.input_ids.shape
```

The important ordering is:

```text
model.forward() -> req.complete_one() -> sampler.sample(...)
```

Use `n` across `model.forward()`; do not drill through model layers during the first pass. Then `r` or `c`.

**Question: does model forward produce logits before sampling, or does sampling happen first?**


In [ ]:
submit(5, "Checkpoint 5: answer forward before sampling.", ["forward", "before"])


## Checkpoint 6: `Scheduler._process_last_data` and decode

Terminal output should include:

```text
TRACE: Scheduler._process_last_data: consume next token
completed phase = prefill
(Pdb)
```

Run:

```text
p last_data[0].batch.phase
p last_data[1].next_tokens_cpu
s
l
n
n
```

In real `scheduler.py`, observe:

```text
next_token = next_tokens_cpu[i]
req.append_host(...)
finished = not req.can_decode
```

With `--max-tokens 2`, the prefill result usually leaves one token to generate. After `r` then `c`, you will see a second sequence of stops with:

```text
phase = decode
```

At the final `_process_last_data`, `remain_len` reaches zero (or EOS appears) and scheduler removes the request and frees/caches resources.

**Question: why do subsequent tokens enter decode, and what must happen to resources after a request ends?**


In [ ]:
submit(6, "Checkpoint 6: include decode and free/release/cache.", ["decode"])


## Completed real PDB trace

You have traced real running mini-sglang through:

```text
LLM.generate
-> Scheduler._process_one_msg
-> PrefillManager.schedule_next_batch
-> Scheduler._prepare_batch
-> Scheduler._forward
-> Engine.forward_batch
-> Scheduler._process_last_data
-> decode iteration
```

This is deliberately offline, TP=1, overlap-disabled, and CUDA-Graph-disabled. It still uses a real Qwen model, real scheduler, real KV cache, real attention backend, and real sampling. API/ZMQ, tensor parallelism, overlap scheduling, and kernel-level debugging are later topics.
